In [1]:

import pandas as pd
import plotly.express as px
import statsmodels.api as sm
import numpy as np


In [2]:
energy=pd.read_csv('energy.csv')

In [3]:
# more than 50% missing values in the columns and unwanted columns

energy.drop(columns=['wind_energy_per_capita','solar_energy_per_capita','renewables_energy_per_capita',
                     'electricity_demand_per_capita','biofuel_energy_per_capita','hydro_energy_per_capita',
                     'gas_energy_per_capita','fossil_energy_per_capita','oil_energy_per_capita',
                     'nuclear_energy_per_capita','net_elec_imports_share_demand','net_elec_imports',
                     'hydro_energy_per_capita','iso_code'] ,inplace=True)

In [4]:
# more than 50% missing values in the rows

unwanted_rows = energy[(energy['year']>=1900)&(energy['year']<1990)].index
energy.drop(unwanted_rows,axis=0,inplace=True)


In [5]:
# Unifying continents,regions,and countries names 


mapping = {'Africa (EI)':'Africa','Africa (EIA)':'Africa','Africa (Ember)':'Africa','Africa (Shift)':'Africa','Other Southern Africa (EI)':'Africa','Other Eastern Africa (EI)':'Africa',
       'Other Middle Africa (EI)':'Africa','Other Western Africa (EI)':'Africa','Other Northern Africa (EI)':'Africa','Other Africa (EI)':'Africa','Western Africa (EI)':'Africa','Eastern Africa (EI)':'Africa', 'Middle Africa (EI)':'Africa',

'Europe (EI)':'Europe','Europe (EIA)':'Europe','Europe (Ember)':'Europe','Europe (Shift)':'Europe','European Union (27)':'Europe','Other Europe (EI)':'Europe','EU28 (Shift)':'Europe','Western Europe (EIA)':'Europe','EU (Ember)':'Europe',

'North America (Ember)':'North America', 'North America (Shift)':'North America','North America (EI)':'North America',

'United States Territories (Shift)':'U.S. Territories','U.S. Territories (EIA)':'U.S. Territories',

'U.S. Pacific Islands (EIA)':'U.S. Pacific Islands','United States Pacific Islands (Shift)':'U.S. Pacific Islands',

 'Middle East (EI)':'Middle East',
       'Middle East (EIA)':'Middle East' , 'Middle East (Ember)': 'Middle East' , 'Middle East (Shift)':'Middle East','Other Middle East (EI)':'Middle East',

'Other South and Central America (EI)': 'South and Central America',  'South and Central America (EI)':'South and Central America',

'Non-OPEC (EI)':'Non-OPEC','Non-OPEC (EIA)':'Non-OPEC',

'Non-OECD (EI)':'Non-OECD','Non-OECD (EIA)':'Non-OECD',

'OECD (EI)':  'OECD', 'OECD (EIA)': 'OECD', 'OECD (Ember)': 'OECD','OECD (Shift)':'OECD',

'OPEC (EI)':'OPEC', 'OPEC (EIA)':  'OPEC' , 'OPEC (Shift)':'OPEC',

'Wake Island (EIA)':'Wake Island','Wake Island (Shift)':'Wake Island',

'Other Asia Pacific (EI)':'Asia Pacific' , 'Other Asia-Pacific (EIA)' : 'Asia Pacific','Asia Pacific (EI)':'Asia Pacific',

'Other North America (EI)':'North America',

'Other South America (EI)':'South America',

'Central and South America (EIA)':'South and Central America', 'Central and South America (EIA)':'South and Central America',
       'Central and South America (Shift)': 'South and Central America','Central and South America (EIA)':'South and Central America',

'Persian Gulf (EIA)':'Persian Gulf',
       'Persian Gulf (Shift)':'Persian Gulf',

'Other CIS (EI)': 'CIS' ,'CIS (EI)':'CIS',

'Asia and Oceania (EIA)':'Asia and Oceania',
       'Asia and Oceania (Shift)':'Asia and Oceania',

'Eurasia (EIA)':'Eurasia','Eurasia (Shift)':'Eurasia',
 'Oceania (Ember)':'Oceania',
'East Germany': 'Germany', 'West Germany':'Germany',
'Asia (Ember)':'Asia', 

'World':'Rest of World','Rest of World (EI)':
'Rest of World'}

energy['country'] = energy['country'].replace(mapping)


In [6]:
energy.drop_duplicates(inplace=True)
energy.reset_index(inplace=True,drop=True)

In [7]:

# Targets are : total_energy_per_capita and electricity_generation

energy_numeric =pd.DataFrame(energy, columns=['population', 'gdp',
       'biofuel_electricity', 'electricity_demand', 'electricity_generation',
       'energy_per_capita', 'energy_per_gdp', 'fossil_electricity',
       'gas_electricity', 'hydro_electricity', 'nuclear_electricity',
       'oil_electricity', 'per_capita_electricity',
       'primary_energy_consumption', 'renewables_electricity',
       'solar_electricity', 'wind_electricity'])

energy_non_numeric =pd.DataFrame(energy, columns=['country', 'year'])



In [8]:
# Checking outliers with a custom function to determine whether to fill missing values by mean or median

def outliers_percentage(series):
    series = series.dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return round(len(outliers) / len(series) * 100, 2)

outliers = {col: outliers_percentage(energy_numeric[col]) for col in energy_numeric.columns}
outliers

{'population': 13.57,
 'gdp': 13.86,
 'biofuel_electricity': 19.39,
 'electricity_demand': 17.82,
 'electricity_generation': 17.57,
 'energy_per_capita': 5.79,
 'energy_per_gdp': 6.52,
 'fossil_electricity': 16.99,
 'gas_electricity': 17.32,
 'hydro_electricity': 17.71,
 'nuclear_electricity': 20.63,
 'oil_electricity': 17.46,
 'per_capita_electricity': 4.28,
 'primary_energy_consumption': 18.26,
 'renewables_electricity': 17.41,
 'solar_electricity': 21.24,
 'wind_electricity': 20.16}

In [9]:
# Imputation for missing values by median

for col in energy_numeric.columns:
    energy_numeric[col].fillna(energy_numeric[col].median(), inplace=True)

C:\Users\ELkayan\AppData\Local\Temp\ipykernel_46528\1193336015.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  energy_numeric[col].fillna(energy_numeric[col].median(), inplace=True)
C:\Users\ELkayan\AppData\Local\Temp\ipykernel_46528\1193336015.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series 

In [10]:
# Vectorized operation: combining all traditional non_renewable energy resources together

energy_numeric['traditional_non_renewables'] = energy_numeric['gas_electricity'] 
+ energy_numeric['fossil_electricity'] + energy_numeric['oil_electricity']


0        370.429985
1        378.650005
2        410.990013
3        435.279995
4        478.170013
            ...    
10147      2.820000
10148      2.510000
10149      2.920000
10150      2.700000
10151           NaN
Length: 10152, dtype: float64

In [11]:
# Vectorized operation: combining energy_per_capita and per_capita_electricity together

energy_numeric['total_energy_per_capita'] = energy_numeric['energy_per_capita']
+ energy_numeric['per_capita_electricity']

0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
            ...    
10147    432.153595
10148    542.500488
10149    556.348572
10150    508.542145
10151           NaN
Name: per_capita_electricity, Length: 10152, dtype: float64

In [12]:
energy_numeric=energy_numeric.drop(columns=['oil_electricity','gas_electricity','fossil_electricity',
                                            'per_capita_electricity','energy_per_capita'])
energy_numeric=round(energy_numeric,2)

In [13]:
energy_clean =pd.concat([energy_non_numeric, energy_numeric], axis=1)

energy_clean.to_csv('energy_clean.csv',index=False)

In [14]:
energy_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10152 entries, 0 to 10151
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   country                     10152 non-null  str    
 1   year                        10152 non-null  int64  
 2   population                  7986 non-null   float64
 3   gdp                         5427 non-null   float64
 4   biofuel_electricity         6107 non-null   float64
 5   electricity_demand          6149 non-null   float64
 6   electricity_generation      7184 non-null   float64
 7   energy_per_gdp              5410 non-null   float64
 8   hydro_electricity           7082 non-null   float64
 9   nuclear_electricity         7330 non-null   float64
 10  primary_energy_consumption  9218 non-null   float64
 11  renewables_electricity      7176 non-null   float64
 12  solar_electricity           7165 non-null   float64
 13  wind_electricity            7008 non-null 

In [15]:
energy_clean.describe(include='float64')

,population,gdp,biofuel_electricity,electricity_demand,electricity_generation,energy_per_gdp,hydro_electricity,nuclear_electricity,primary_energy_consumption,renewables_electricity,solar_electricity,wind_electricity,traditional_non_renewables,total_energy_per_capita
count,7.986000e+03,5.427000e+03,6107.000000,6149.000000,7184.000000,5410.000000,7082.000000,7330.000000,9218.000000,7176.000000,7165.000000,7008.000000,6618.000000,7812.000000
mean,1.206456e+08,6.998285e+11,12.190516,637.755760,810.697027,1.542567,127.949016,110.014229,4568.187777,180.664833,12.278486,26.366995,183.189095,26286.660797
std,5.940046e+08,5.088740e+12,51.641144,2531.199181,2708.590038,1.464960,410.098015,386.597457,16432.214728,633.452762,82.900408,135.324643,594.594980,40059.570249
min,1.776000e+03,2.571720e+08,0.000000,0.000000,0.000000,0.080000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.124908e+05,1.944098e+10,0.000000,1.560000,1.920000,0.750000,0.050000,0.000000,7.140000,0.140000,0.000000,0.000000,0.000000,3019.030000
50%,6.387486e+06,6.400624e+10,0.020000,12.530000,19.900000,1.170000,2.900000,0.000000,73.810000,3.960000,0.000000,0.000000,2.180000,13028.560000
75%,2.526980e+07,2.959861e+11,0.920000,94.360000,151.582500,1.877500,24.100000,5.207500,717.637500,33.010000,0.100000,0.630000,44.247500,35686.162500
max,8.161973e+09,1.301126e+14,704.810000,30938.180000,30938.180000,23.020000,4419.950000,2778.760000,176737.090000,9845.460000,2128.370000,2505.350000,6900.360000,651836.560000


#  Univariate Analysis         
 

**Distribution of Energy Resources**

In [16]:

for col in energy_clean[['renewables_electricity','traditional_non_renewables',
       'solar_electricity', 'wind_electricity','biofuel_electricity',
       'hydro_electricity','nuclear_electricity'
       ]]:
    fig=px.histogram(energy_clean,x=col,nbins=10,title=f'Distribution of {col}',
                     color_discrete_sequence=['steelblue'],marginal='rug')
    fig.show()

**Energy Overtime (Time Series Analysis)**

In [17]:
for col in energy_clean[['solar_electricity', 'wind_electricity',
                         'biofuel_electricity','hydro_electricity','nuclear_electricity',
                         'renewables_electricity','traditional_non_renewables']]:
    total=energy_clean.groupby('year')[col].sum().reset_index()
    fig= px.line(total,x='year',y=col,
                   color_discrete_sequence=['steelblue'])
    fig.show()

**All kinds of renewable energy generation have generally increased overtime.**


**Despite the  growth of renewable energy , traditional non-renewable(oil,gas,fossil fuel) energy resources and nuclear energy still contribute largely in  global energy generation.**


# Bivariate Analysis 

**renewable energy VS. electricity_generation**

In [18]:
for col in energy_clean[['solar_electricity', 'wind_electricity','renewables_electricity',
                         'biofuel_electricity','hydro_electricity']]:
    fig= px.scatter(energy_clean,x='electricity_generation',y=col,
                   trendline= 'ols',color_discrete_sequence=['cornflowerblue'])
    fig.show()

All kinds of renewable energy resources correlates  positively with total electricity generation (increased generation).  Hydro power has the strongest correlation and the most stable growth

**energy resource VS. total_energy_per_capita**

In [19]:
# renewable energy VS. total_energy_per_capita

for col in energy_clean[['solar_electricity', 'wind_electricity','renewables_electricity',
                         'biofuel_electricity','hydro_electricity']]:
    fig= px.scatter(energy_clean,x='total_energy_per_capita',y=col,
                   trendline='ols',color_discrete_sequence=['slateblue'])
    fig.show()

There is no direct relation between renewable energy  and the amount of individual share of energy 

**renewable energy VS. energy_per_gdp**

In [20]:
for col in energy_clean[['solar_electricity', 'wind_electricity','renewables_electricity',
                         'biofuel_electricity','hydro_electricity']]:
    fig = solar_gdp= px.scatter(energy_clean,x='energy_per_gdp',y=col,
                   trendline='ols',color_discrete_sequence=['slateblue'])
    fig.show()

There is no direct relation between the renewable energy and energy_per_gdp

# Multivariate Analysis

In [21]:
# renewable energy  VS. energy access indicators(electricity_generation,total_energy_per_capita,energy_per_gdp)

corr_matrix = px.imshow(energy_clean[[
            'electricity_generation','total_energy_per_capita','energy_per_gdp',
            'hydro_electricity', 'biofuel_electricity',
            'solar_electricity', 'wind_electricity','renewables_electricity',
            ]].corr(),text_auto = True, color_continuous_scale=px.colors.sequential.Blues_r)
corr_matrix

Renewable energy has a weak impact/indirect relationship with the individual share of energy and energy_per_gdp, but correlates positively with electricity generation

In [22]:
# renewable energy  VS. GDP

gdp_matrix = px.imshow(energy_clean[['gdp',
                        'hydro_electricity', 'biofuel_electricity',
                        'solar_electricity', 'wind_electricity','renewables_electricity',
                        ]].corr(),text_auto = True,color_continuous_scale=px.colors.sequential.Purples_r)
gdp_matrix



Renewable energy correlates weakly with GDP

# Income-based Analysis

In [23]:
income = energy_clean[(energy_clean['country'] == 'Low-income countries')|
                      (energy_clean['country'] =='Lower-middle-income countries' )|
                      (energy_clean['country'] =='Upper-middle-income countries' )|
                      (energy_clean['country'] == 'High-income countries')]

In [24]:
for col in income[['renewables_electricity','traditional_non_renewables']]:
    fig= px.histogram(income,x=col,facet_col='country',
                      color_discrete_sequence=['royalblue'],title=f'Distribution of {col}')
    fig.show()

In [25]:
# Energy Resources Central Tendency 

for col in income[['renewables_electricity',
       'solar_electricity', 'wind_electricity','biofuel_electricity',
       'hydro_electricity',
       ]]:
    fig=px.box(income,y=col,facet_col='country',color_discrete_sequence=['steelblue'])
    fig.show()
    

In [26]:

for col in income[['renewables_electricity',
       'solar_electricity', 'wind_electricity','biofuel_electricity',
       'hydro_electricity',
       ]]:
    fig=px.bar(income,y=col,x='country',color_discrete_sequence=['steelblue'])
    fig.show()

**High-income countries produce more renewable energy than low and middle-income countries**

**Hydro power and wind energy are the top two kinds of renewable energy adapted in high-income and upper-middle-income countries**


# Renewable Energy by Continent

In [27]:
continent = energy_clean[(energy_clean['country'] == 'Africa')|
                      (energy_clean['country'] =='Australia')|
                      (energy_clean['country'] == 'Asia')|
                      (energy_clean['country'] == 'Europe')|
                      (energy_clean['country'] == 'North America')|
                      (energy_clean['country'] == 'South America')
                      ] 

In [28]:

for col in continent[['renewables_electricity',
       'solar_electricity', 'wind_electricity','biofuel_electricity',
       'hydro_electricity',
       ]]:
    fig=px.bar(continent,y=col,x='country',color_discrete_sequence=['steelblue'])
    fig.show()


## Top Regions/Countries in Renewable Energy

In [29]:
filters = ['Africa',  'Asia', 'Asia and Oceania', 'Australia',  'Low-income countries',
           'Lower-middle-income countries','Upper-middle-income countries','High-income countries',
           'Australia and New Zealand (EIA)','G20 (Ember)', 'G7 (Ember)', 'OECD', 'OPEC',
           'Non-OPEC', 'North America','Europe','Rest of World', 'South and Central America',
           'Latin America and Caribbean (Ember)','Non-OECD', 'Non-OPEC','South America']
    

filtered_country=energy_clean[~energy_clean['country'].isin(filters)]

In [30]:
for col in filtered_country[['biofuel_electricity','hydro_electricity',
                        'renewables_electricity','solar_electricity','wind_electricity']]:
    
    df=filtered_country.groupby('country')[col].sum().reset_index().sort_values(by=col,ascending=False)
    pie=px.pie(df.head(5),values=col,names='country',
                   color_discrete_sequence=px.colors.sequential.Blues_r,title=f'Top 5 in {col}')
    pie.show()
                 

In [33]:
%%writefile app.py

import pandas as pd
import plotly.express as px
import streamlit as st
import statsmodels as sm


df= pd.read_csv('energy_clean.csv')

st.set_page_config(page_title='Energy',layout='wide')
st.title('Energy Dashboard')


# Year and Country Filters

st.sidebar.header('Filters')

selected_country=st.sidebar.selectbox('select country',df['country'].unique())

min_year=int(df['year'].min())
max_year= int(df['year'].max())
year_filter= st.sidebar.slider('year range',min_year,max_year,
                                (min_year,max_year))


# Applying filters to create Interactive Dashboard

filtered_df=df

if selected_country:

    filtered_df = filtered_df[
    (filtered_df['country']==(selected_country))]
    filtered_df=filtered_df[(filtered_df['year'].between(year_filter[0], year_filter[1]))]
                        
                       
    st.dataframe(filtered_df)

# Dividing the Webage into Multiple parts

page=st.radio('choose an option',['Navigate','Analysis'])


if page=='Navigate':
    st.subheader('Data General View')
    

elif page=='Analysis':
    
    

    tab1,tab2,tab3,tab4,tab5,tab6,tab7=st.tabs(['Overview','Renewable Energy Over Time','Income-based Analysis',
    'Renewable Energy VS. Electricity','Renewable Energy by Continent','Energy Matrix','Summary'])

    with tab1:
    
        st.subheader('General KPIs')
        col1,col2=st.columns(2,gap='large')
        col3,col4=st.columns(2,gap='large')
        col5,col7=st.columns(2,gap='large')

        def format_number(value):
            if value >= 1e9:
                return f"{value/1e9:.2f}B"
            elif value >= 1e6:
                return f"{value/1e6:.2f}M"
            elif value >= 1e3:
                return f"{value/1e3:.2f}K"
            else:
                return f"{value:.2f}"


        avg_energy = round(filtered_df['electricity_generation'].mean(),2)
        avg_energy_per_capita = round(filtered_df['total_energy_per_capita'].mean(),2)
        avg_gdp =round(filtered_df['gdp'].mean(),2)
        total_pop = round(filtered_df['population'].sum(),2)
        electricity_demand =round(filtered_df['electricity_demand'].mean(), 2)
        energy_per_gdp=round(filtered_df['energy_per_gdp'].mean(), 2)

        col1.metric('average energy generation',format_number(avg_energy))
        col2.metric('average energy_per_capita',format_number(avg_energy_per_capita) )
        col3.metric('average GDP',format_number(avg_gdp))
        col4.metric('total population',format_number(total_pop))
        col5.metric('average energy_demand',format_number(electricity_demand))
        col7.metric('average energy_per_gdp',energy_per_gdp )

##############################################################################################################
    with tab2:

        st.subheader('Time Series Analysis')
        col8,col9,col10=st.columns(3,gap='large')
        col11,col12=st.columns(2,gap='large')

        
        df1=df.groupby('year')[['solar_electricity', 'wind_electricity',
                            'biofuel_electricity','hydro_electricity',
                            'renewables_electricity']].sum().reset_index()

        with col8:

            fig1 = px.line(df1,y='renewables_electricity',x='year')
            st.plotly_chart(fig1,use_container_width=True)

        with col9:
            fig2=px.line(df1,y='solar_electricity',x='year')
            st.plotly_chart(fig2,use_container_width=True)

        with col10:
            fig3=px.line(df1,y='hydro_electricity',x='year')
            st.plotly_chart(fig3,use_container_width=True)
            

        with col11:
            fig4=px.line(df1,y='wind_electricity',x='year')
            st.plotly_chart(fig4,use_container_width=True)
            
        with col12:
            fig5=px.line(df1,y='biofuel_electricity',x='year')
            st.plotly_chart(fig5,use_container_width=True)

        st.markdown('All kinds of renewable energy have generally increased over time')


#######################################################################################################
    with tab3:

        income = df[(df['country'] == 'Low-income countries')|
                        (df['country'] =='Lower-middle-income countries' )|
                        (df['country'] =='Upper-middle-income countries' )|
                        (df['country'] == 'High-income countries')]
        

        col13,col14,col15=st.columns(3,gap='large')
        col16,col17=st.columns(2,gap='large')
        col18,col19=st.columns(2,gap='large')

        with col13:
            fig6=px.bar(income,y='renewables_electricity',x='country')
            st.plotly_chart(fig6,use_container_width=True)


        with col14:
            fig7=px.bar(income,y='solar_electricity',x='country')
            st.plotly_chart(fig7,use_container_width=True)


        with col15:
            fig8=px.bar(income,y='wind_electricity',x='country')
            st.plotly_chart(fig8,use_container_width=True)


        with col16:
            fig9=px.bar(income,y='hydro_electricity',x='country')
            st.plotly_chart(fig9,use_container_width=True)

        with col17:
            fig10=px.bar(income,y='biofuel_electricity',x='country')
            st.plotly_chart(fig10,use_container_width=True)

        st.markdown('High-income countries produce more renewable energy than low and middle-income countries')
        st.markdown('Hydro power and wind energy are the top two kinds of renewable energy adapted in high-income and upper-middle-income countries')
############################################################################################################################
    with tab4:
       
        st.subheader('Renewable Energy VS. Electricity Generation')

        col20,col21,col22=st.columns(3,gap='large')
        col23,col24=st.columns(2,gap='large')
        

        with col20:                
            fig13= px.scatter(df,x='electricity_generation',y='solar_electricity',
                   trendline='ols')
            st.plotly_chart(fig13,use_container_width=True)

        with col21:                
            fig14= px.scatter(df,x='electricity_generation',y='wind_electricity',
                   trendline='ols')
            st.plotly_chart(fig14,use_container_width=True)
        
        with col22:                
            fig15= px.scatter(df,x='electricity_generation',y='renewables_electricity',
                   trendline='ols')
            st.plotly_chart(fig15,use_container_width=True)
        
        with col23:                
            fig16= px.scatter(df,x='electricity_generation',y='hydro_electricity',
                   trendline='ols')
            st.plotly_chart(fig16,use_container_width=True)

        with col24:                
            fig17= px.scatter(df,x='electricity_generation',y='biofuel_electricity',
                   trendline='ols')
            st.plotly_chart(fig17,use_container_width=True)

        st.markdown('All kinds of renewable energy resources correlates positively with total electricity generation (increased generation)')
        st.markdown('Hydro power has the strongest correlation and the most stable growth')               
##############################################################################################################################
    with tab5:

        col25,col26,col27=st.columns(3,gap='large')
        col28,col29=st.columns(2,gap='large')
     
        continent = df[(df['country'] == 'Africa')|
                      (df['country'] =='Australia')|
                      (df['country'] == 'Asia')|
                      (df['country'] == 'Europe')|
                      (df['country'] == 'North America')|
                      (df['country'] == 'South America')
                      ] 

        with col25:
            fig18=px.bar(continent,y='renewables_electricity',x='country')
            st.plotly_chart(fig18,use_container_width=True)

        with col26:
            fig19=px.bar(continent,y='solar_electricity',x='country')
            st.plotly_chart(fig19,use_container_width=True)

           

        with col27:
            fig20=px.bar(continent,y='wind_electricity',x='country')
            st.plotly_chart(fig20,use_container_width=True)



        with col28:
            fig21=px.bar(continent,y='biofuel_electricity',x='country')
            st.plotly_chart(fig21,use_container_width=True)

          
        with col29:
            fig22=px.bar(continent,y='hydro_electricity',x='country')
            st.plotly_chart(fig22,use_container_width=True)

            
    
######################################################################################################
    with tab6:
       
        st.subheader('renewable energy VS. energy access indicators')
        
        corr_matrix = px.imshow(df[[
                            'electricity_generation','total_energy_per_capita','energy_per_gdp',
                            'hydro_electricity', 'biofuel_electricity','solar_electricity', 
                            'wind_electricity','renewables_electricity'
                            ]].corr(),text_auto = True,color_continuous_scale=px.colors.sequential.Blues_r)
                            
        st.plotly_chart(corr_matrix,use_container_width=True)
   
        st.markdown("""Renewable energy has a weak relationship with the individual share of energy and energy_per_gdp, 
        but correlates positively with electricity generation""")
######################################################################################################
    with tab7:
        st.markdown("""
        # Main Conclusion

        Global Renewable Energy Trends:
        
        All kinds of renewable energy have generally increased over time, with 
        accelerated growth after 2000. Wind energy and Hydropower are the dominant 
        resources that have grown more rapidly than other renewables.

        However, despite this growth, traditional non-renewable resources(oil, gas, and fossil fuels) 
        and nuclear energy still contribute largely in global energy generation.
        """)

        st.markdown("""
        **Hydropower is the top renewable resource that maintains a 
        strong positive correlation with total electricity generation.**
        """)

    
        st.markdown("""
        Economic Impact:        
            Higher-income countries generally adopt renewable energy 
            than lower-income countries.
        """)
                    
        

Overwriting app.py


In [32]:
! streamlit run app.py

^C
